# Tariff Impact Explorer — Analytical Notebook

This notebook is the required analytical companion to the Streamlit app `app.py`. It walks through the full workflow from raw data to findings, in the order a marker would expect: problem statement, data loading with source provenance, cleaning decisions (including a unit-correction that is easy to miss), exploratory analysis, a worked example of the event-study methodology, and closing findings.

**Submission track:** Track 4 — Interactive Data Analysis Tool (ACC102 Mini Assignment).

## 1. Problem & User

**Analytical problem.** When a new US tariff is announced, how did financial and commodity markets react? Static news reports describe events one at a time and cannot answer follow-up questions such as *"which asset moved most in the ten trading days after Liberation Day?"* or *"did soybeans or USD/CNY have the larger reaction to the April 2025 145% escalation?"*.

**Target user.** Finance journalists, junior trade analysts, and undergraduate business students who need quick evidence for a brief, article, or case study — not a formal academic study.

**Why Track 4 fits.** The question is exploratory by nature: the user wants to change the event, the asset list, and the window length and see the effect immediately. An interactive Streamlit tool serves this pattern better than a static report.

## 2. Data Loading & Source

**Source.** *US Tariff & Trade War Impact Dataset (2018–Present)* by BELBIN BENO, published on Kaggle. The dataset aggregates Yahoo Finance, FRED, and US Census series.

**Source URL.** https://www.kaggle.com/datasets/belbino/us-tariff-and-trade-war-impact-dataset-2018-present

**Access date.** 2026-04-18

**Files used (all under `data/`):**
- `market_reaction.csv` — daily prices for S&P 500, Shanghai Composite, DXY, USD/CNY, WTI crude, steel futures, aluminium futures, soybeans. Starts **2020-01-02**.
- `tariff_rates.csv` — 15 major tariff announcements (2018-03-08 Section 232 steel through the 2025 Liberation Day / China-145% / 90-day-pause cycle).
- `trade_balance.csv` — monthly US trade balance.

**File deliberately excluded.** `tariff_news_headlines.csv` covers only 10 days (2026-04-06 to 2026-04-15) and mixes low-credibility sources. See §3 for the detailed exclusion reasoning.

**Four tariff events dropped as out-of-range.** Four announcements in `tariff_rates.csv` (both Section 232 rows on 2018-03-08, the 2018-07-06 China tech tariffs, and the 2019-09-01 consumer-goods round) predate the 2020-01-02 market-data start. We cannot build a real event window for them — they are removed in the analysis below. The app surfaces the 11 remaining events.

**On data validation.** The assignment brief (§5) requires students to judge whether data are *"sufficiently reliable for educational analysis"*. The dataset's underlying sources (Yahoo Finance close prices, FRED/Census trade figures, USTR / Federal Register announcement dates) are standard references for this kind of exploratory work. Spot-checks of individual S&P 500 closes and USD/CNY values against Yahoo Finance's public site are consistent with the dataset. For an educational Track 4 product this level of verification is appropriate; a trading-grade or policy-grade use case would demand direct pulls from each primary source.

In [1]:
import sys
from pathlib import Path

# Make `src/` importable regardless of where the notebook is launched from.
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd

from src.data_loader import load_market_reaction, load_tariff_rates, load_trade_balance
from src.cleaning import clean_market_reaction, clean_trade_balance, null_report
from src.event_study import (
    compute_event_window,
    summarize_event,
    build_sensitivity_matrix,
)

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 140)

2026-04-18 08:42:09.432 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-04-18 08:42:09.433 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-04-18 08:42:09.433 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


In [2]:
market_raw = load_market_reaction()
tariffs_full = load_tariff_rates()
trade_raw = load_trade_balance()

# Drop tariff events that predate the market data (see §2).
market_start = market_raw['date'].min()
tariffs = tariffs_full.loc[tariffs_full['date'] >= market_start].reset_index(drop=True)
dropped = len(tariffs_full) - len(tariffs)

print(f"market_reaction.csv : {market_raw.shape[0]:>5} rows, {market_raw.shape[1]} cols, "
      f"{market_raw['date'].min().date()} -> {market_raw['date'].max().date()}")
print(f"tariff_rates.csv    : {len(tariffs_full):>5} raw rows -> {len(tariffs)} in-range (dropped {dropped} pre-{market_start.date()})")
print(f"trade_balance.csv   : {trade_raw.shape[0]:>5} rows, {trade_raw.shape[1]} cols")

2026-04-18 08:42:09.490 
  command:

    streamlit run /Volumes/ssd/projects/NSP/books/tariffimpackexplorer/tariff-impact-explorer/.venv/lib/python3.14/site-packages/ipykernel_launcher.py [ARGUMENTS]


2026-04-18 08:42:09.491 No runtime found, using MemoryCacheStorageManager


2026-04-18 08:42:09.508 No runtime found, using MemoryCacheStorageManager


2026-04-18 08:42:09.513 No runtime found, using MemoryCacheStorageManager


market_reaction.csv :  1581 rows, 9 cols, 2020-01-02 -> 2026-04-17
tariff_rates.csv    :    15 raw rows -> 11 in-range (dropped 4 pre-2020-01-02)
trade_balance.csv   :    74 rows, 2 cols


## 3. Cleaning

Three cleaning concerns, handled explicitly so they are auditable:

1. **Null values in daily market data.** Chinese-market holidays leave gaps in `shanghai_composite`, plus a handful of one-off gaps in `usd_cny` and `steel_futures`. We forward-fill so every trading day carries a value; forward-fill is the conventional choice because the next available close is the most faithful representation of "last known price".
2. **Unit mis-label on trade balance.** The CSV column is named `us_trade_balance_bn`, but the values are in **millions** of USD, not billions. We confirm this below with a sanity check against the reported monthly US trade deficit, then rename the column and add a billions version.
3. **News file.** Excluded; documented in §2.

In [3]:
# Null counts BEFORE cleaning.
print('Null counts in market_reaction (before cleaning):')
print(null_report(market_raw).to_string(index=False))

Null counts in market_reaction (before cleaning):
            column  null_count
              date           0
             sp500           0
shanghai_composite         112
               dxy           0
           usd_cny           1
     crude_oil_wti           0
     steel_futures           1
  aluminum_futures           0
          soybeans           0


In [4]:
market = clean_market_reaction(market_raw)
print('Null counts in market_reaction (after forward-fill):')
print(null_report(market).to_string(index=False))

Null counts in market_reaction (after forward-fill):
            column  null_count
              date           0
             sp500           0
shanghai_composite           0
               dxy           0
           usd_cny           0
     crude_oil_wti           0
     steel_futures           0
  aluminum_futures           0
          soybeans           0


In [5]:
# Unit-correction sanity check on the trade-balance file.
print('BEFORE correction (first 3 rows):')
print(trade_raw.head(3).to_string(index=False))
print()
print(
    'The BEA reported a monthly US goods-and-services deficit of about '
    'USD 43.6 billion for January 2020. The raw value -43562 therefore has '
    'to be in millions, not billions. We divide by 1000 to expose billions.'
)

BEFORE correction (first 3 rows):
      date  us_trade_balance_bn
2020-01-01               -43562
2020-02-01               -39931
2020-03-01               -42374

The BEA reported a monthly US goods-and-services deficit of about USD 43.6 billion for January 2020. The raw value -43562 therefore has to be in millions, not billions. We divide by 1000 to expose billions.


In [6]:
trade = clean_trade_balance(trade_raw)
print('AFTER correction (first 3 rows):')
print(trade.head(3).to_string(index=False))

AFTER correction (first 3 rows):
      date  us_trade_balance_mn  us_trade_balance_bn
2020-01-01               -43562              -43.562
2020-02-01               -39931              -39.931
2020-03-01               -42374              -42.374


## 4. Exploratory Analysis

In [7]:
# Descriptive statistics on the daily market series.
market.drop(columns=['date']).describe().round(2)

,sp500,shanghai_composite,dxy,usd_cny,crude_oil_wti,steel_futures,aluminum_futures,soybeans
count,1581.00,1581.00,1581.00,1581.00,1581.00,1581.00,1581.00,1581.00
mean,4687.30,3317.77,100.04,0.14,70.26,953.91,2395.24,1233.24
std,1108.76,314.70,5.37,0.01,18.70,357.46,429.05,236.09
min,2237.40,2660.17,89.44,0.14,-37.63,447.00,1452.00,821.75
25%,3925.43,3084.93,96.32,0.14,61.50,705.00,2182.50,1029.00
50%,4412.53,3284.83,100.03,0.14,71.65,862.00,2397.50,1209.25
75%,5572.85,3493.05,104.15,0.15,80.58,1085.00,2599.50,1426.75
max,7126.06,4182.59,114.11,0.16,123.70,1945.00,3873.00,1769.00


In [8]:
# Correlation of daily returns across assets. Returns (pct_change) are more
# informative than price-level correlation because price levels are trending.
returns = market.drop(columns=['date']).pct_change().dropna()
returns.corr().round(2)

,sp500,shanghai_composite,dxy,usd_cny,crude_oil_wti,steel_futures,aluminum_futures,soybeans
sp500,1.00,0.13,-0.17,-0.03,0.12,0.01,0.13,0.08
shanghai_composite,0.13,1.00,-0.11,0.08,0.03,0.03,0.24,0.06
dxy,-0.17,-0.11,1.00,-0.02,-0.01,-0.02,-0.21,-0.11
usd_cny,-0.03,0.08,-0.02,1.00,-0.00,0.03,0.04,0.00
crude_oil_wti,0.12,0.03,-0.01,-0.00,1.00,0.01,0.10,0.08
steel_futures,0.01,0.03,-0.02,0.03,0.01,1.00,0.06,0.03
aluminum_futures,0.13,0.24,-0.21,0.04,0.10,0.06,1.00,0.19
soybeans,0.08,0.06,-0.11,0.00,0.08,0.03,0.19,1.00


In [9]:
# Tariff events by country / region.
tariffs.groupby('country').agg(
    n_events=('headline', 'count'),
    avg_rate=('tariff_rate_pct', 'mean'),
    max_rate=('tariff_rate_pct', 'max'),
).round(1)

,n_events,avg_rate,max_rate
country,,,
Canada,1,25.0,25.0
China,7,51.1,145.0
Global,2,50.0,90.0
Mexico,1,25.0,25.0


## 5. Event-Study Methodology — Worked Example

**Event.** 2025-04-02 — *Liberation Day*: Trump's 10% baseline tariff on all imports. We pick this event because (a) it falls inside the market-data range, (b) it is the most widely discussed tariff event of the 2025 cycle, and (c) it is broad enough that we expect a visible reaction across multiple assets (S&P 500, USD/CNY, Shanghai Composite) rather than in one sector alone.

**Window.** `(-5, 10)` trading days around the announcement.

**Expectation.** A broad 10% tariff on all imports is a demand-side shock for risk assets and a dollar / CNY event. We expect the S&P 500 to react negatively in the post-window, USD/CNY to move (likely higher = weaker CNY), and sector commodities like soybeans to reflect China-retaliation risk. We are not making formal forecasts — we want to see whether the size and direction of each move is consistent with the market narrative.

In [10]:
event = tariffs.loc[tariffs['date'] == '2025-04-02'].iloc[0]
print('Event row:')
print(event.to_string())

window = compute_event_window(
    market, tariffs,
    event_headline=event['headline'],
    window=(-5, 10),
)
window.round(2)

Event row:
date                                                 2025-04-02 00:00:00
country                                                           Global
product_category                                               All Goods
tariff_rate_pct                                                     10.0
announcement_source                                       Liberation Day
headline               Liberation Day: Trump announces 10% baseline t...


,date,trading_day_offset,sp500,shanghai_composite,dxy,usd_cny,crude_oil_wti,steel_futures,aluminum_futures,soybeans
0,2025-03-26,-5,100.73,100.55,100.71,100.18,97.13,98.91,107.06,97.23
1,2025-03-27,-4,100.39,100.71,100.25,100.18,97.50,97.27,104.75,98.76
2,2025-03-28,-3,98.41,100.04,100.22,100.09,96.72,97.27,103.93,99.37
3,2025-03-31,-2,98.96,99.57,100.39,100.10,99.68,97.27,102.78,98.57
4,2025-04-01,-1,99.33,99.95,100.43,100.18,99.29,97.81,101.44,100.46
5,2025-04-02,0,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
6,2025-04-03,1,95.16,99.76,98.32,100.03,93.36,99.78,97.03,98.25
7,2025-04-04,2,89.47,99.76,99.24,100.03,86.45,101.86,93.37,94.90
8,2025-04-07,3,89.27,92.43,99.47,99.84,84.65,101.64,92.93,95.48
9,2025-04-08,4,87.86,93.89,99.18,99.47,83.08,100.55,92.57,96.43


In [11]:
# Self-check: the event-day row must have trading_day_offset == 0
# and every asset column normalised to 100.
event_day = window.loc[window['trading_day_offset'] == 0]
asset_cols = [c for c in window.columns if c not in {'date', 'trading_day_offset'}]
assert len(event_day) == 1, 'There should be exactly one event-day row.'
assert np.allclose(event_day[asset_cols].values, 100.0), 'Event-day normalisation failed.'
print('Self-check passed: event-day row is normalised to 100 across all assets.')

Self-check passed: event-day row is normalised to 100 across all assets.


In [12]:
summarize_event(window).round(4)

,asset,pre_event_return,post_event_return,post_event_volatility,max_drawdown
0,sp500,-0.0072,-0.0697,0.0433,-0.1214
1,shanghai_composite,-0.0055,-0.0221,0.0258,-0.0757
2,dxy,-0.0071,-0.0427,0.0095,-0.0427
3,usd_cny,-0.0018,-0.0058,0.0029,-0.0112
4,crude_oil_wti,0.0296,-0.1289,0.0386,-0.1692
5,steel_futures,0.0110,0.0273,0.0087,-0.0129
6,aluminum_futures,-0.0659,-0.0237,0.0223,-0.0846
7,soybeans,0.0285,0.0090,0.0166,-0.0510


## 6. Findings & Link to App

The following observations come directly from the event-study and sensitivity-matrix computations in §5 and the cell below. They are intentionally descriptive — this is exploratory work, not formal statistical inference on 11 events.

1. **Liberation Day (2025-04-02).** The worked example above produces a visible reaction across the S&P 500 and USD/CNY in the first 10 trading days after the announcement — the direction and magnitude are readable directly from the summary table.
2. **Sensitivity is asset-specific.** The same event does not move every asset by the same sign or magnitude; the heatmap in `app.py` makes this pattern visible across all 11 in-range events.
3. **Announcement-day alignment matters.** Some tariff dates fall on weekends or holidays. The `compute_event_window` function handles this by snapping to the nearest trading day on or after the announcement; without that alignment the event-day row would be missing and every downstream metric would shift by one.
4. **Data-range alignment matters too.** Four tariff events predate the market-data start (2020-01-02); we drop them rather than silently anchor their 'event day' on January 2020. This is easy to miss and would otherwise produce four meaningless rows in the heatmap.
5. **Daily-only granularity limits causal claims.** Large intraday reactions to an overnight announcement are collapsed into a single close-to-close return, so the measured effect understates the true announcement reaction.

See `app.py` for interactive exploration of all 11 in-range events across any subset of the 8 asset classes.

In [13]:
# Cross-event sensitivity: 10-trading-day cumulative returns across all events.
matrix = build_sensitivity_matrix(market, tariffs, window_length=10)
matrix.round(3) * 100  # shown as percentages

,sp500,shanghai_composite,dxy,usd_cny,crude_oil_wti,steel_futures,aluminum_futures,soybeans
2020-01-15 — US reduces China tariffs to 7.5% under Phase 1 tra,-0.2,-3.7,0.7,-0.5,-9.8,-0.3,-2.7,-5.7
2022-10-07 — US expands chip export controls and tech tariffs o,3.1,0.5,-0.7,-1.4,-8.2,1.9,-3.1,2.1
2024-05-14 — Biden raises tariffs on Chinese EVs to 100%,0.4,-1.1,0.1,-0.2,1.6,-0.5,8.5,1.3
2024-05-14 — Biden raises tariffs on Chinese solar cells and ch,0.4,-1.1,0.1,-0.2,1.6,-0.5,8.5,1.3
2025-02-01 — Trump adds blanket 10% tariff on all Chinese impor,2.3,2.3,-1.8,-0.9,-1.8,4.2,0.8,-1.9
2025-03-04 — Trump raises China tariffs to 20% cumulative,-2.8,3.2,-2.4,0.7,-2.0,2.8,0.8,2.9
2025-03-04 — Trump imposes 25% tariffs on Canadian imports,-2.8,3.2,-2.4,0.7,-2.0,2.8,0.8,2.9
2025-03-04 — Trump imposes 25% tariffs on Mexican imports,-2.8,3.2,-2.4,0.7,-2.0,2.8,0.8,2.9
2025-04-02 — Liberation Day: Trump announces 10% baseline tarif,-7.0,-2.2,-4.3,-0.6,-12.9,2.7,-2.4,0.9
2025-04-09 — US raises tariffs on China to 145% after retaliati,0.5,3.5,-3.5,0.7,0.7,2.6,10.3,4.0
